In [1]:
import numpy as np
import os, sys, random, math
import copy
import sklearn
import sqlite3
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import ScalarFormatter
import datetime
from dateutil import parser
import pickle
import csv

clusters = ["AMS20PrdApp19-tround.sqlite","LON23PrdApp01-troundgrt5m.sqlite","BLAPrdApp19-troundgrt5m.sqlite","LVL01PrdApp05-troundgrt5m.sqlite","BN9PrdApp18-troundgrt5m.sqlite","SG2PrdApp35-troundgrt5m.sqlite","DSM08PrdApp05-troundgrt5m.sqlite","SYD21PrdApp07-troundgrt5m.sqlite","YTO21PrdApp05-troundgrt5m.sqlite","DUB24PrdApp09-troundgrt5m.sqlite"]
# cluster = "LON23PrdApp01-troundgrt5m.sqlite"
cluster = "AMS20PrdApp19-tround.sqlite"
print(cluster)

AMS20PrdApp19-tround.sqlite


In [2]:
class VM:
    def __init__(self):
        self.vm_id = -1
        self.node_id = -1
        self.cores = -1
        self.memory = -1
        self.nic = -1
        self.rss = list()
        self.ssd = -1
        self.max_mem_ts = list()
        self.avg_mem_ts = list()
        self.max_cpu_ts = list()  # set to None
        self.avg_cpu_ts = list()  # set to None
        self.start_time = datetime.datetime(2024,1,1,1,1)
        self.end_time = datetime.datetime(2020,1,1,1,1)
        self.instance_role = ""  # set to None
        self.subscription_id = ""  # set to None

In [3]:
with open(f"traces/{cluster}.pkl", "rb") as f:
    all_vms, node_to_vms, node_to_machine, vm_type_sz, machine_sz = pickle.load(f)
print(len(all_vms))
print(len(node_to_vms))

34029
1415


In [4]:
import csv

with open("topologies/random_L96_X8_N4_Y192.csv", "r") as fp:
    reader = csv.reader(fp)
    rows = [row for row in reader][1:]
server_dev_tup_list = [(int(row[0]), int(row[1])) for row in rows]

num_mhds = len(set(tup[1] for tup in server_dev_tup_list))
num_hosts = len(set(tup[0] for tup in server_dev_tup_list))
matrix = [[0 for _ in range(num_mhds)] for _ in range(num_hosts)]
for host, dev in server_dev_tup_list:
    matrix[host][dev] = 1
random_96_matrix = matrix


In [5]:
import csv

with open("topologies/AG16x6_expander_quads_r5_sym_fixed.csv", "r") as fp:
    reader = csv.reader(fp)
    rows = [row for row in reader][1:]
server_dev_tup_list = [(int(row[0]), int(row[1])) for row in rows]

num_mhds = len(set(tup[1] for tup in server_dev_tup_list))
num_hosts = len(set(tup[0] for tup in server_dev_tup_list))
matrix = [[0 for _ in range(num_mhds)] for _ in range(num_hosts)]
for host, dev in server_dev_tup_list:
    matrix[host][dev] = 1
AG16x6_expander_quads_r5_sym_matrix = matrix

In [6]:
import csv

with open("topologies/AG16x4_latin3_r5_sym.csv", "r") as fp:
    reader = csv.reader(fp)
    rows = [row for row in reader][1:]
server_dev_tup_list = [(int(row[0]), int(row[1])) for row in rows]

num_mhds = len(set(tup[1] for tup in server_dev_tup_list))
num_hosts = len(set(tup[0] for tup in server_dev_tup_list))
matrix = [[0 for _ in range(num_mhds)] for _ in range(num_hosts)]
for host, dev in server_dev_tup_list:
    matrix[host][dev] = 1
AG16x4_latin3_r5_sym_matrix = matrix

In [7]:
import csv

with open("topologies/bibd_25.csv", "r") as fp:
    reader = csv.reader(fp)
    rows = [row for row in reader][1:]
server_dev_tup_list = [(int(row[0]), int(row[1])) for row in rows]

num_mhds = len(set(tup[1] for tup in server_dev_tup_list))
num_hosts = len(set(tup[0] for tup in server_dev_tup_list))
matrix = [[0 for _ in range(num_mhds)] for _ in range(num_hosts)]
for host, dev in server_dev_tup_list:
    matrix[host][dev] = 1
bibd25_matrix = matrix

# Greedy & Optimal

In [8]:
def greedy_alloc(cxl_mem, mhd_list, cur_cxl_mem_vec):
    alloc_arr = np.zeros((len(cur_cxl_mem_vec),))
    cur_cxl_mem_vec = np.maximum(cur_cxl_mem_vec, np.zeros((len(cur_cxl_mem_vec),)))
    original_cxl_mem = cxl_mem
    while cxl_mem > 0:
        min_mhd = mhd_list[0]
        for mhd in mhd_list:
            if cur_cxl_mem_vec[mhd] < cur_cxl_mem_vec[min_mhd]:
                min_mhd = mhd
        num_min_mhd = 0
        next_min_mhd = None
        for mhd in mhd_list:
            if cur_cxl_mem_vec[mhd] == cur_cxl_mem_vec[min_mhd]:
                num_min_mhd += 1
            if cur_cxl_mem_vec[mhd] > cur_cxl_mem_vec[min_mhd] and (next_min_mhd is None or cur_cxl_mem_vec[mhd] < cur_cxl_mem_vec[next_min_mhd]):
                next_min_mhd = mhd
        assert num_min_mhd > 0
        assert next_min_mhd is None or cur_cxl_mem_vec[min_mhd] < cur_cxl_mem_vec[next_min_mhd]

        min_val = cur_cxl_mem_vec[min_mhd]
        if next_min_mhd is None or cxl_mem <= num_min_mhd * (cur_cxl_mem_vec[next_min_mhd] - cur_cxl_mem_vec[min_mhd]):
            for mhd in mhd_list:
                 if cur_cxl_mem_vec[mhd] == min_val:
                     alloc_arr[mhd] += cxl_mem / num_min_mhd
                     cur_cxl_mem_vec[mhd] += cxl_mem / num_min_mhd
            cxl_mem = 0
        else:
            to_alloc = cur_cxl_mem_vec[next_min_mhd] - cur_cxl_mem_vec[min_mhd]
            for mhd in mhd_list:
                if cur_cxl_mem_vec[mhd] == min_val:
                    alloc_arr[mhd] += to_alloc
                    cur_cxl_mem_vec[mhd] += to_alloc
            cxl_mem -= to_alloc * num_min_mhd

    return alloc_arr

In [9]:
import datetime as _dt
import numpy as np

def greedy_pooling_simulation(
    node_to_M,                   # dict: node -> node_in_pod_id (0..pod_size-1)
    M,                           # list/np.array shape [pod_size, num_mhd] (0/1 usable matrix or capacities)
    host_mem_dist_pct_list=None,
):
    pod_size = len(M)
    assert len(node_to_M) == pod_size, "node_to_M size must match rows of M"
    assert pod_size > 0, "Empty pod"
    num_mhd = len(M[0])
    assert num_mhd > 0, "No MHDs"
    mem_idx = 1

    # 1) Compute pod resource totals and time range robustly
    pod_rss = np.zeros(4, dtype=float)
    n_start_time = _dt.datetime.max
    n_end_time = _dt.datetime.min

    any_vm = False
    for cur_node in node_to_M.keys():
        pod_rss += np.asarray(machine_sz[node_to_machine[cur_node]], dtype=float)
        for cur_vmkey in node_to_vms.get(cur_node, []):
            cur_vm = all_vms[cur_vmkey]
            any_vm = True
            if cur_vm.start_time < n_start_time:
                n_start_time = cur_vm.start_time
            if cur_vm.end_time > n_end_time:
                n_end_time = cur_vm.end_time

    if not any_vm:
        # Nothing to simulate
        return 0.0

    # Convert times to 5-minute tick indexes using a consistent base
    base = _dt.datetime(n_start_time.year, n_start_time.month, n_start_time.day, n_start_time.hour, n_start_time.minute)
    def to_tick(t):
        return int((t - base).total_seconds() // 300)

    pod_start_ts = to_tick(n_start_time)
    pod_end_ts = to_tick(n_end_time)
    pod_dur = pod_end_ts - pod_start_ts + 1
    if pod_dur <= 0:
        return 0.0

    # 2) HOTFIX: filter out VMs that would exceed per-node memory
    vmkey_to_skip = set()
    for cur_node in node_to_M.keys():
        # events per tick
        alloc_events = [[] for _ in range(pod_dur)]
        dealloc_events = np.zeros((pod_dur,), dtype=float)
        node_rss = np.asarray(machine_sz[node_to_machine[cur_node]], dtype=float)

        for cur_vmkey in node_to_vms.get(cur_node, []):
            cur_vm = all_vms[cur_vmkey]
            vm_start = to_tick(cur_vm.start_time) - pod_start_ts
            vm_end = to_tick(cur_vm.end_time) - pod_start_ts
            if vm_end < 0:  # skip invalid VMs
                vmkey_to_skip.add(cur_vmkey)
                continue

            vm_rss_vec = np.asarray(cur_vm.rss, dtype=float)
            mem = float(vm_rss_vec[mem_idx])

            alloc_events[vm_start].append((cur_vmkey, vm_end + 1, mem))
            if vm_end + 1 < pod_dur:
                dealloc_events[vm_end + 1] += mem

        cur_mem = 0.0
        for ts in range(pod_dur):
            cur_mem -= dealloc_events[ts]
            if cur_mem < 0:
                cur_mem = 0.0  # guard against tiny numeric drift
            for vmkey, end_ts, mem in alloc_events[ts]:
                cur_mem += mem
                if cur_mem > node_rss[mem_idx]:
                    # reject and undo
                    cur_mem -= mem
                    vmkey_to_skip.add(vmkey)
                    if end_ts < pod_dur:
                        dealloc_events[end_ts] -= mem

    # 3) Per-pod simulation with greedy allocation
    alloc_events = [[] for _ in range(pod_dur)]
    for cur_node, node_in_pod_id in node_to_M.items():
        for cur_vmkey in node_to_vms.get(cur_node, []):
            if cur_vmkey in vmkey_to_skip:
                continue
            cur_vm = all_vms[cur_vmkey]
            vm_start = to_tick(cur_vm.start_time) - pod_start_ts
            vm_end = to_tick(cur_vm.end_time) - pod_start_ts

            vm_rss_vec = np.asarray(cur_vm.rss, dtype=float)
            alloc_events[vm_start].append((node_in_pod_id, vm_end + 1, vm_rss_vec))

    cur_cxl_mem_vec = np.zeros((num_mhd,), dtype=float)
    max_cxl_mem_vec = np.zeros((num_mhd,), dtype=float)
    dealloc_events = np.zeros((pod_dur, num_mhd), dtype=float)

    for i, cur_event_list in enumerate(alloc_events):
        cur_cxl_mem_vec -= dealloc_events[i, :]
        # numerical safety
        cur_cxl_mem_vec = np.maximum(cur_cxl_mem_vec, 0.0)

        for node_in_pod_id, dealloc_time, cur_event in cur_event_list:
            mem = float(cur_event[mem_idx])
            if mem <= 0:
                continue

            # list of usable MHDs for this node
            mhd_list = [mhd for mhd in range(num_mhd) if M[node_in_pod_id][mhd] != 0]
            assert len(mhd_list) != 0

            alloc_vec = np.asarray(greedy_alloc(mem, mhd_list, cur_cxl_mem_vec), dtype=float)
            assert alloc_vec.shape == (num_mhd,), "greedy_alloc must return shape (num_mhd,)"
            if np.any(alloc_vec < -1e-9):
                raise ValueError("Negative allocation from greedy_alloc")

            tot = float(np.sum(alloc_vec))
            if not (mem - 1.0 <= tot <= mem + 1.0):
                # relax if rounding; otherwise treat as error
                assert False

            cur_cxl_mem_vec += alloc_vec

            # ensure dealloc strictly after i
            assert dealloc_time > i
            if dealloc_time < pod_dur:
                dealloc_events[dealloc_time, :] += alloc_vec

        max_cxl_mem_vec = np.maximum(max_cxl_mem_vec, cur_cxl_mem_vec)

    denom = float(pod_rss[mem_idx])
    if denom <= 0:
        return 0.0

    return float(np.max(max_cxl_mem_vec)) * num_mhd / denom


In [10]:
import numpy as np
from collections import deque

# ---------- Max-flow (Dinic) ----------
class _Dinic:
    __slots__ = ("N","g","level","it")
    def __init__(self, N):
        self.N = N
        self.g = [[] for _ in range(N)]
        self.level = [0]*N
        self.it = [0]*N
    def _add_edge(self, u, v, c):
        self.g[u].append([v, float(c), len(self.g[v])])
        self.g[v].append([u, 0.0, len(self.g[u])-1])
    def add_edge(self, u, v, c): self._add_edge(u, v, c)
    def bfs(self, s, t, eps=1e-12):
        self.level = [-1]*self.N
        dq = deque([s]); self.level[s] = 0
        while dq:
            u = dq.popleft()
            for v, cap, _ in self.g[u]:
                if cap > eps and self.level[v] < 0:
                    self.level[v] = self.level[u] + 1
                    dq.append(v)
        return self.level[t] >= 0
    def dfs(self, u, t, f, eps=1e-12):
        if u == t: return f
        gi = self.g[u]
        for i in range(self.it[u], len(gi)):
            self.it[u] = i
            v, cap, rev = gi[i]
            if cap > eps and self.level[u] + 1 == self.level[v]:
                d = self.dfs(v, t, min(f, cap), eps)
                if d > eps:
                    gi[i][1] -= d
                    self.g[v][rev][1] += d
                    return d
        return 0.0
    def max_flow(self, s, t):
        flow = 0.0; INF = 1e100
        while self.bfs(s, t):
            self.it = [0]*self.N
            while True:
                f = self.dfs(s, t, INF)
                if f <= 1e-12: break
                flow += f
        return flow

# ---------- Connected components on bipartite graph ----------
def _bipartite_components(M_bin):
    """
    M_bin: bool array (n x m). Returns list of (rows_idx, cols_idx) for each component.
    """
    n, m = M_bin.shape
    # Build adjacency for rows<->cols
    row_to_cols = [np.nonzero(M_bin[i])[0] for i in range(n)]
    col_to_rows = [np.nonzero(M_bin[:,j])[0] for j in range(m)]
    row_seen = np.zeros(n, dtype=bool)
    col_seen = np.zeros(m, dtype=bool)
    comps = []

    for r0 in range(n):
        if row_seen[r0]: continue
        if row_to_cols[r0].size == 0:
            # isolated row: its own component (no usable MHDs)
            row_seen[r0] = True
            comps.append((np.array([r0], dtype=int), np.array([], dtype=int)))
            continue

        # BFS/DFS alternating between row and col sets
        rows = []
        cols = []
        rq = deque([r0])
        row_seen[r0] = True
        while rq:
            r = rq.popleft()
            rows.append(r)
            for c in row_to_cols[r]:
                if not col_seen[c]:
                    col_seen[c] = True
                    cols.append(c)
                    # add all rows connected to this column
                    for rr in col_to_rows[c]:
                        if not row_seen[rr]:
                            row_seen[rr] = True
                            rq.append(rr)
        comps.append((np.array(rows, dtype=int), np.array(cols, dtype=int)))

    # Also check for isolated columns that haven't been touched (no incident rows)
    for c0 in range(m):
        if not col_seen[c0]:
            col_seen[c0] = True
            comps.append((np.array([], dtype=int), np.array([c0], dtype=int)))

    return comps

# ---------- Per-component solver (binary search + flow) ----------
_state_cache = {}  # warm-start LB/UB per component topology

def _component_peak(b_sub, M_sub, key_bytes, iters=30):
    """
    b_sub: (n_sub,) demands
    M_sub: (n_sub, m_sub) bool mask
    key_bytes: unique bytes for caching (e.g., M_sub.tobytes())
    """
    n, m = M_sub.shape
    total = float(b_sub.sum())
    if n == 0 or m == 0 or total <= 0.0:
        return 0.0

    deg = M_sub.sum(axis=1)
    if np.any((deg == 0) & (b_sub > 0)):
        return float('inf')  # infeasible

    # Lower/upper bounds
    per_row_lb = np.max(np.divide(b_sub, np.where(deg > 0, deg, 1),
                                  where=deg>0, out=np.zeros_like(b_sub)))
    LB = max(total / m, per_row_lb)
    UB = total

    if key_bytes in _state_cache:
        pLB, pUB = _state_cache[key_bytes]
        LB = max(LB, pLB); UB = min(UB, pUB)
        if LB > UB: LB, UB = pLB, pUB

    # Build base graph: s(0) -> rows(1..n), rows->cols, cols->t(1+n..1+n+m-1)
    s = 0; first_r = 1; first_c = 1 + n; t_sink = 1 + n + m - 1
    baseG = _Dinic(1 + n + m)
    for i in range(n):
        baseG.add_edge(s, first_r + i, b_sub[i])
    INF = 1e100
    for i in range(n):
        u = first_r + i
        for j in np.nonzero(M_sub[i])[0]:
            v = first_c + j
            baseG.add_edge(u, v, INF)

    def feasible(cap):
        # clone base and add column->sink edges with capacity cap
        G2 = _Dinic(baseG.N)
        for u in range(baseG.N):
            G2.g[u] = [e.copy() for e in baseG.g[u]]
        for j in range(m):
            G2.add_edge(first_c + j, t_sink, cap)
        return G2.max_flow(s, t_sink) >= total - 1e-8

    lo, hi = LB, UB
    for _ in range(iters):  # fewer iterations suffice in practice
        mid = 0.5*(lo + hi)
        if feasible(mid): hi = mid
        else: lo = mid

    _state_cache[key_bytes] = (lo, hi)
    return float(hi)

# ---------- Public API: find_optimal with component splitting ----------
def find_optimal(node_cxl_arr, M):
    """
    Minimizes the maximum per-MHD load with adjacency mask M by splitting into
    connected components and solving each independently. Returns the optimal peak
    *per MHD* (same semantics as the CVXPY version).
    """
    b = np.asarray(node_cxl_arr, dtype=float).reshape(-1)
    M_bin = np.asarray(M, dtype=bool)
    n, m = M_bin.shape
    assert b.shape[0] == n, "Length of node_cxl_arr must match rows of M"

    # Remove zero-demand rows early (pure speed)
    keep_rows = ~(np.isclose(b, 0.0))
    if keep_rows.any() and not keep_rows.all():
        b = b[keep_rows]
        M_bin = M_bin[keep_rows, :]
        n, m = M_bin.shape

    # Remove isolated zero-degree rows/columns early (also speed)
    # (Rows with deg=0 and demand=0 are safe to drop; cols with deg=0 don't bind.)
    row_deg = M_bin.sum(axis=1)
    col_deg = M_bin.sum(axis=0)
    row_keep = (row_deg > 0) | (b > 0)
    col_keep = (col_deg > 0)
    if (not row_keep.all()) or (not col_keep.all()):
        M_bin = M_bin[row_keep][:, col_keep]
        b = b[row_keep]
        n, m = M_bin.shape

    if n == 0 or m == 0 or b.sum() <= 0.0:
        return 0.0

    # Split into components and solve each
    comps = _bipartite_components(M_bin)
    t_star = 0.0
    for rows_idx, cols_idx in comps:
        if rows_idx.size == 0:
            # no rows ⇒ no demand in this component
            continue
        b_sub = b[rows_idx]
        M_sub = M_bin[np.ix_(rows_idx, cols_idx)]
        if b_sub.sum() <= 0.0:
            continue
        key_bytes = M_sub.tobytes()  # stable topology key for warm-start caching
        t_comp = _component_peak(b_sub, M_sub, key_bytes)
        if t_comp > t_star:
            t_star = t_comp

    return float(t_star)

import datetime as _dt
import numpy as np

def optimal_pooling_simulation(
    node_to_M,   # dict: node_id -> node_in_pod_id (0..pod_size-1)
    M,           # list/np.array shape [pod_size, num_mhd] (0/1 usable matrix)
    window=1,
):
    """
    Optimal (per-tick) CXL pooling simulation for a single pod.
    Assumes 100% of VM memory goes to CXL (no local DRAM usage accounting).

    Expects globals:
      - node_to_machine: dict[node_id] -> machine_type
      - machine_sz: dict[machine_type] -> iterable of 4 resource capacities (index 1 is memory)
      - node_to_vms: dict[node_id] -> list of vm_keys
      - all_vms: dict[vm_key] -> object with .start_time, .end_time, .rss (len>=2; index 1 is memory)
      - find_optimal(node_demands, M) -> float (max per-MHD load, prior to scaling by num_mhd)
    """
    pod_size = len(M)
    assert pod_size > 0, "Empty pod matrix M"
    assert len(node_to_M) == pod_size, "node_to_M size must match rows of M"
    num_mhd = len(M[0])
    assert num_mhd > 0, "No MHDs"
    mem_idx = 1

    # 1) Compute pod totals and time window
    pod_rss = np.zeros(4, dtype=float)
    n_start_time = _dt.datetime.max
    n_end_time = _dt.datetime.min
    any_vm = False

    for cur_node in node_to_M.keys():
        pod_rss += np.asarray(machine_sz[node_to_machine[cur_node]], dtype=float)
        for cur_vmkey in node_to_vms.get(cur_node, []):
            cur_vm = all_vms[cur_vmkey]
            any_vm = True
            if cur_vm.start_time < n_start_time:
                n_start_time = cur_vm.start_time
            if cur_vm.end_time > n_end_time:
                n_end_time = cur_vm.end_time

    if not any_vm:
        return 0.0

    # Convert times to 5-minute ticks relative to a stable base
    base = _dt.datetime(n_start_time.year, n_start_time.month, n_start_time.day,
                        n_start_time.hour, n_start_time.minute)

    def to_tick(t):
        return int((t - base).total_seconds() // 300)

    pod_start_ts = to_tick(n_start_time)
    pod_end_ts = to_tick(n_end_time)
    pod_dur = pod_end_ts - pod_start_ts + 1
    if pod_dur <= 0:
        return 0.0

    # 2) HOTFIX: filter VMs that would overflow per-node DRAM (kept for data hygiene parity)
    vmkey_to_skip = set()
    for cur_node in node_to_M.keys():
        alloc_events = [[] for _ in range(pod_dur)]
        dealloc_events = np.zeros((pod_dur,), dtype=float)
        node_rss = np.asarray(machine_sz[node_to_machine[cur_node]], dtype=float)

        for cur_vmkey in node_to_vms.get(cur_node, []):
            cur_vm = all_vms[cur_vmkey]
            vm_start = to_tick(cur_vm.start_time) - pod_start_ts
            vm_end   = to_tick(cur_vm.end_time)   - pod_start_ts
            if vm_end < 0:
                vmkey_to_skip.add(cur_vmkey)
                continue

            mem = float(np.asarray(cur_vm.rss, dtype=float)[mem_idx])
            alloc_events[vm_start].append((cur_vmkey, vm_end + 1, mem))
            if vm_end + 1 < pod_dur:
                dealloc_events[vm_end + 1] += mem

        cur_mem = 0.0
        for ts in range(pod_dur):
            cur_mem -= dealloc_events[ts]
            if cur_mem < 0:
                cur_mem = 0.0
            for vmkey, end_ts, mem in alloc_events[ts]:
                cur_mem += mem
                if cur_mem > node_rss[mem_idx]:
                    # reject & undo
                    cur_mem -= mem
                    vmkey_to_skip.add(vmkey)
                    if end_ts < pod_dur:
                        dealloc_events[end_ts] -= mem

    # 3) Build per-node CXL demand (all memory goes to CXL)
    diff_cxl = np.zeros((pod_dur, pod_size), dtype=float)  # difference array per node index
    for cur_node, node_in_pod_id in node_to_M.items():
        for cur_vmkey in node_to_vms.get(cur_node, []):
            if cur_vmkey in vmkey_to_skip:
                continue
            cur_vm = all_vms[cur_vmkey]
            vm_start = to_tick(cur_vm.start_time) - pod_start_ts
            vm_end   = to_tick(cur_vm.end_time)   - pod_start_ts
            if vm_end < 0:
                continue
            mem = float(np.asarray(cur_vm.rss, dtype=float)[mem_idx])
            if mem <= 0:
                continue
            diff_cxl[vm_start, node_in_pod_id] += mem
            if vm_end + 1 < pod_dur:
                diff_cxl[vm_end + 1, node_in_pod_id] -= mem

    # 4) Sweep time, apply optimal placement each tick
    node_cxl_arr = np.zeros((pod_size,), dtype=float)
    max_cxl_mem = 0.0
    ts_count = 0
    for ts in range(pod_dur):
        node_cxl_arr += diff_cxl[ts, :]
        if np.max(diff_cxl[ts, :]) <= 0:
            continue
        ts_count += 1
        if ts_count % window == 0:
            print(f"\r{ts}/{pod_dur}", end="")
            # find_optimal returns max per-MHD load (before scaling); keep your original scaling
            cur_cxl_mem = float(find_optimal(node_cxl_arr, M)) * num_mhd
            if cur_cxl_mem > max_cxl_mem:
                max_cxl_mem = cur_cxl_mem
    print("\r", end="")

    denom = float(pod_rss[mem_idx])
    if denom <= 0:
        return 0.0

    return max_cxl_mem / denom

In [11]:
def generate_pod_to_nodes(pod_size, seed):
    random.seed(seed)

    node_list = [int(node_id) for node_id in node_to_vms.keys()]
    random.shuffle(node_list)

    pod_to_nodes = dict()
    for i in range(len(node_list) // pod_size):
        pod_to_nodes[i] = list(node_list[i * pod_size:(i + 1) * pod_size])
    return pod_to_nodes

In [12]:
def expand_M_to_all_nodes(M, pod_to_nodes):
    pod_list = sorted(list(pod_to_nodes.keys()))

    num_nodes_per_pod = len(pod_to_nodes[pod_list[0]])
    assert len(M) == num_nodes_per_pod
    num_mhd_per_pod = len(M[0])
    num_pods = len(pod_to_nodes)

    node_to_M = dict()
    expanded_M = list()
    for pod in pod_list:
        local_M = list()
        for node in range(num_nodes_per_pod):
            M_row = list()
            node_to_M[pod_to_nodes[pod][node]] = len(expanded_M) + len(local_M)
            for mhd_pod in range(num_pods):
                if mhd_pod == pod:
                    M_row.extend(M[node])
                else:
                    M_row.extend([0 for _ in range(num_mhd_per_pod)])
            local_M.append(M_row)
        expanded_M.extend(local_M)
    return node_to_M, expanded_M

In [13]:
def remove_ones(matrix, ratio, seed=None):
    """
    Remove exactly floor(total_ones * ratio) ones from the matrix while ensuring
    each row that originally had at least one '1' keeps at least one '1'.
    The 'kept' one per such row is chosen uniformly at random.

    Raises:
        ValueError if the requested removal count is infeasible.
    """
    if seed is not None:
        random.seed(seed)

    # Collect all 1-positions and per-row 1-positions
    ones_positions = [(i, j)
                      for i, row in enumerate(matrix)
                      for j, val in enumerate(row) if val == 1]
    total_ones = len(ones_positions)

    # Quick exit: nothing to remove
    target_remove = int(total_ones * ratio)
    if total_ones == 0 or target_remove == 0:
        return copy.deepcopy(matrix)

    # For each row that has >=1 one, randomly choose one to keep
    must_keep = set()
    rows_with_ones = 0
    for i, row in enumerate(matrix):
        row_ones = [(i, j) for j, val in enumerate(row) if val == 1]
        if row_ones:
            rows_with_ones += 1
            must_keep.add(random.choice(row_ones))  # random keep per row

    # Max removable given the per-row keep constraint
    max_removable = total_ones - rows_with_ones
    if target_remove > max_removable:
        raise ValueError(
            f"Infeasible: requested to remove {target_remove} ones "
            f"but at least one '1' must remain in each of the {rows_with_ones} "
            f"rows that originally had a '1' (max removable = {max_removable})."
        )

    # Choose exactly target_remove positions to remove from the remaining pool
    removable = [pos for pos in ones_positions if pos not in must_keep]
    positions_to_remove = set(random.sample(removable, target_remove))

    # Build new matrix
    new_matrix = copy.deepcopy(matrix)
    for i, j in positions_to_remove:
        new_matrix[i][j] = 0

    return new_matrix

In [14]:
matrix = random_96_matrix
result_list = list()
for i in range(50):
    print(f"\rIteration={i}", end="")
    pod_to_nodes = generate_pod_to_nodes(len(matrix), 10086 + i)
    node_to_M, expanded_M = expand_M_to_all_nodes(matrix, pod_to_nodes)
    ratio = greedy_pooling_simulation(node_to_M, expanded_M)
    result_list.append(1 - ratio)
print(f"avg:\t{np.mean(result_list)}\nmin:\t{np.min(result_list)}\nmax:\t{np.max(result_list)}\nstd:\t{np.std(result_list)}")

Iteration=49avg:	0.2662140797535641
min:	0.19134324744962317
max:	0.30544608384544036
std:	0.021286723951622608


In [ ]:
matrix = AG16x6_expander_quads_r5_sym_matrix
result_list = list()
for i in range(20):
    print(f"Iteration={i}")
    pod_to_nodes = generate_pod_to_nodes(len(matrix), 10086 + i)
    node_to_M, expanded_M = expand_M_to_all_nodes(matrix, pod_to_nodes)
    ratio = optimal_pooling_simulation(node_to_M, expanded_M, window=1)
    result_list.append(1 - ratio)
print(f"avg:\t{np.mean(result_list)}\nmin:\t{np.min(result_list)}\nmax:\t{np.max(result_list)}\nstd:\t{np.std(result_list)}")

Iteration=0
10/2304

In [ ]:
matrix = AG16x6_expander_quads_r5_sym_matrix
for failure_ratio in [0.01, 0.02, 0.03, 0.04, 0.05, 0.1]:
    result_list = list()
    for i in range(50):
        print(f"\rIteration={i}", end="")
        pod_to_nodes = generate_pod_to_nodes(len(matrix), 10086 + i)
        node_to_M, expanded_M = expand_M_to_all_nodes(matrix, pod_to_nodes)
        expanded_M = remove_ones(expanded_M, failure_ratio, 0xbeef + i)

        ratio = greedy_pooling_simulation(node_to_M, expanded_M)
        result_list.append(1 - ratio)
    print(f"\nfailure_ratio={failure_ratio}")
    print(f"avg:\t{np.mean(result_list)}\nmin:\t{np.min(result_list)}\nmax:\t{np.max(result_list)}\nstd:\t{np.std(result_list)}")

In [ ]:
matrix = AG16x6_expander_quads_r5_sym_matrix

In [ ]:
for failure_ratio in [0.01, 0.02, 0.03, 0.04, 0.05]:
    result_list = list()
    for i in range(20):
        print(f"\rIteration={i}", end="")
        pod_to_nodes = generate_pod_to_nodes(len(matrix), 10086 + i)
        node_to_M, expanded_M = expand_M_to_all_nodes(matrix, pod_to_nodes)
        expanded_M = remove_ones(expanded_M, failure_ratio, 0xbeef + i)
        ratio = greedy_pooling_simulation(node_to_M, expanded_M)
        result_list.append(ratio)
    print(f"\nFailure={failure_ratio}:")
    print(f"\tavg:\t{np.mean(result_list)}\n\tmin:\t{np.min(result_list)}\n\tmax:\t{np.max(result_list)}\n\tstd:\t{np.std(result_list)}")